-------------------------------------------------------------------------------------------------------
# EUSS Post-Retrofit Measure Packages: MP8, MP9, MP10
-------------------------------------------------------------------------------------------------------
- MP8: Whole Home Electrification (High Efficiency)
- MP9: Whole-Home Electrification + Basic Enclosure Upgrade
- MP10: Whole-Home Electrification + Enhanced Enclosure Upgrade

-------------------------------------------------------------------------------------------------------
# TARE MODEL SCENARIOS
-------------------------------------------------------------------------------------------------------
- Pre-IRA Scenario:
    - NREL End-Use Savings Shapes Database: Measure Package 8/9/10
    - AEO2023 No Inflation Reduction Act
    - Cambium 2021 MidCase
      
- IRA-Reference Scenario:
    - NREL End-Use Savings Shapes Database: Measure Package 8/9/10
    - AEO2023 REFERENCE CASE - HDD and Fuel Price Projections
    - Cambium 2022 and 2023 MidCase

In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
import os
from IPython import get_ipython
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Project configuration
from config import PROJECT_ROOT

# Model constants - explicit imports for clarity
from cmu_tare_model.constants import (
    VERBOSE, 
    RCM_MODELS, 
    CR_FUNCTIONS,
    PRINT_DEBUG,
    PRINT_VERBOSE_DATAFRAMES
)
from cmu_tare_model.utils.discounting import (
    PRIVATE_DISCOUNT_RATE_COLS, 
    PRIVATE_DISCOUNT_RATE_SHORT_KEYS
)

# Data loading utility
from cmu_tare_model.utils.load_exported_results_to_df import load_model_run_output, load_measure_package_data

# =============================================================================
# MATPLOTLIB/SEABORN CONFIGURATION
# =============================================================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
plt.close('all')
%matplotlib inline

sns.set_theme(font='sans-serif', style='darkgrid')

# =============================================================================
# PROJECT ROOT AND TIMESTAMP SETUP
# =============================================================================
# Get the current datetime
start_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Format the name of the exported results file using the location ID
result_export_time = datetime.now()
model_run_date_time = result_export_time.strftime("%Y-%m-%d_%H-%M")

print(f"""
PROJECT_ROOT: {PROJECT_ROOT}

Start Time: {start_time}
Model Run Timestamp: {model_run_date_time}

""")

In [ ]:
# Select whether to begin new run or visualize existing model outputs
while True:
    try:
        start_new_model_run = str(input("""
Would you like to begin a new simulation or visualize output results from a previous model run? Please enter one of the following:
Y. I'd like to start a new model run.
N. I'd like to visualize output results from a previous model run.""")).upper()

        print(f"Enter the following input: {start_new_model_run}")

        if start_new_model_run == 'Y':
            print(f"Formatted date for use in file name: {model_run_date_time}")

            # Relative path to the file from the project root
            relative_path = os.path.join("cmu_tare_model", "model_scenarios", "tare_run_simulation_v2_2.ipynb")

            # Construct the absolute path to the file
            file_path = os.path.join(PROJECT_ROOT, relative_path)
            print(f"File path: {file_path}")

            # Storing Result Outputs in output_results folder
            output_folder_path = os.path.join(PROJECT_ROOT, "cmu_tare_model", "output_results")
            print(f"Result outputs will be exported here: {output_folder_path}")

            # On Windows, to avoid any path-escape quirks, convert backslashes to forward slashes
            file_path = file_path.replace("\\", "/")

            print(f"Running file: {file_path}")

            # iPthon magic command to run a .py file and import variables into the current IPython session
            if os.path.exists(file_path):
                get_ipython().run_line_magic('run', f'-i {file_path}')  # If your path has NO spaces, no quotes needed.
            else:
                print(f"File not found: {file_path}")

            break  # Exit the loop if input is 'Y'
            
        elif start_new_model_run == 'N':
            # Enter the date time of the model run in the following format: YYYY-MM-DD_HH-MM
            model_run_date_time = str(input("Enter the date time of the model run in the following format YYYY-MM-DD_HH-MM: "))
            location_id = str(input("Enter the location ID used in the model run (e.g., 'National' or 'PA'): "))
            
            # Load model run results
            print(f"Loading model run results for location ID: {location_id} and timestamp: {model_run_date_time}")

            # Storing Result Outputs in output_results folder
            output_folder_path = os.path.join(PROJECT_ROOT, "cmu_tare_model", "output_results")
            print(f"Past model run results will be loaded from here: {output_folder_path}")
            
            break  # Exit the loop if input is 'N'
        
        else:
            print("Invalid input. Please enter 'Y' or 'N'.")
    
    except Exception as e:
        print("An error occurred:", e)
        print("Please try again.")

In [ ]:
if VERBOSE:
    print(f"""
    ====================================================================================================================================================================
    LOAD SCENARIO DATA
    ====================================================================================================================================================================
    The load_model_run_output function loads scenario data from a specified folder and date. Additional details are provided below:
        
    Documentation for the load_model_run_output function:
    {load_model_run_output.__doc__}

    -----------------------------------------------------------------------------------------------
    LOADING SCENARIO DATA ...

    These parameters are common to all function calls:
    Output folder path: {output_folder_path}
    Model run date time: {model_run_date_time}
    """)

-------------------------------------------------------------------------------------------------------
# Baseline Scenario: Measure Package 0 (MP0)
-------------------------------------------------------------------------------------------------------

In [ ]:
# =======================================================================================================
# Baseline Scenario: Measure Package 0 (MP0)
# =======================================================================================================
columns_to_string = {16: str, 19: str, 20: str, 21: str}
menu_mp = 0

df_outputs_baseline_home = load_model_run_output(
    results_category='summary_baseline',
    menu_mp=menu_mp,
    output_folder_path=output_folder_path,
    location_id=location_id,
    results_export_formatted_date=model_run_date_time,
    columns_to_string=columns_to_string,
    use_chunked_loading=True,
    chunk_size=10000
)

-------------------------------------------------------------------------------------------------------
# Basic Retrofit: Measure Package 8 (MP8)
-------------------------------------------------------------------------------------------------------

In [ ]:
# =============================================================================
# LOAD MODEL RESULTS: MP8, MP9, MP10
# =============================================================================
# Common column type specification
columns_to_string = {16: str, 19: str, 20: str, 21: str}

# Load all measure packages
DATAFRAMES_MP8 = load_measure_package_data(8, output_folder_path, location_id, model_run_date_time, columns_to_string)
DATAFRAMES_MP9 = load_measure_package_data(9, output_folder_path, location_id, model_run_date_time, columns_to_string)
DATAFRAMES_MP10 = load_measure_package_data(10, output_folder_path, location_id, model_run_date_time, columns_to_string)

# Convenience mapping for downstream code
DATAFRAMES_BY_MP = {
    8: DATAFRAMES_MP8,
    9: DATAFRAMES_MP9,
    10: DATAFRAMES_MP10
}

# =============================================================================
# DATAFRAME EXTRACTION FOR VISUALIZATIONS
# =============================================================================
# Direct dictionary access with short keys - no helper function needed
# Structure: DATAFRAMES[discount_rate][rcm_model]

# (Using FIXED_BASE as the primary discount rate for most visualizations)
df_outputs_mp8_ap2_FIXED_BASE = DATAFRAMES_MP8['fixed_base']['ap2']
df_outputs_mp8_easiur_FIXED_BASE = DATAFRAMES_MP8['fixed_base']['easiur']
df_outputs_mp8_inmap_FIXED_BASE = DATAFRAMES_MP8['fixed_base']['inmap']

# Sensitivity Analyses DataFrames for MP8
df_outputs_mp8_inmap_FIXED_LOW = DATAFRAMES_MP8['fixed_low']['inmap']
df_outputs_mp8_inmap_FIXED_HIGH = DATAFRAMES_MP8['fixed_high']['inmap']
df_outputs_mp8_inmap_VARIABLE = DATAFRAMES_MP8['variable']['inmap']

# Dataframes used for MP9 and MP10 ASHP comparisons
df_outputs_mp9_ap2_FIXED_BASE = DATAFRAMES_MP9['fixed_base']['ap2']
df_outputs_mp9_easiur_FIXED_BASE = DATAFRAMES_MP9['fixed_base']['easiur']
df_outputs_mp9_inmap_FIXED_BASE = DATAFRAMES_MP9['fixed_base']['inmap']

df_outputs_mp10_ap2_FIXED_BASE = DATAFRAMES_MP10['fixed_base']['ap2']
df_outputs_mp10_easiur_FIXED_BASE = DATAFRAMES_MP10['fixed_base']['easiur']
df_outputs_mp10_inmap_FIXED_BASE = DATAFRAMES_MP10['fixed_base']['inmap']

# CLIMATE CHANGE AND PUBLIC HEALTH IMPACTS

In [ ]:
from cmu_tare_model.utils.data_visualization import print_summary_stats
from cmu_tare_model.utils.data_visualization_boxplots import create_subplot_grid_boxplot
from cmu_tare_model.utils.data_visualization_histograms import create_subplot_grid_histogram, print_positive_percentages_complete

if VERBOSE:
    print(f"""  
    ====================================================================================================================================================================
    UNCERTAINTY ANALYSIS VISUALIZATION
    ====================================================================================================================================================================

    --------------------------------------------------------
    SUMMARY STATISTICS TABLE
    --------------------------------------------------------
    data_visualization.py file contains the documentation for the print_summary_stats function.

    --------------------------------------------------------
    SUBPLOT GRID OF BOXPLOTS
    --------------------------------------------------------
    data_visualization_boxplots.py file contains the documentation for the create_subplot_grid_boxplot function.
        
    --------------------------------------------------------
    SUBPLOT GRID OF HISTOGRAMS
    --------------------------------------------------------
    data_visualization_histograms.py file contains the documentation for the create_subplot_grid_histogram function.
        
    --------------------------------------------------------------------------------------------------------------------------------------------------------------------
    """)

## HEALTH IMPACT: 3 Reduced Complexity Models x 2 CR Functions

In [ ]:
# =============================================================================
# HEALTH IMPACT VISUALIZATIONS: RCM × CR-Function Sensitivity
# =============================================================================
scenario_prefix = 'iraRef_mp8_'
category = 'heating'
lower_percentile = 0.5
upper_percentile = 99.5

# Store figures for later reference
health_npv_figures = {}

for cr_function in CR_FUNCTIONS:  # ['acs', 'h6c']
    print(f"\n{'='*60}")
    print(f"FIGURE: MONETIZED HEALTH IMPACT ({cr_function.upper()} CR-FUNCTION)")
    print(f"{'='*60}")
    
    fig = create_subplot_grid_boxplot(
        dataframes=[
            df_outputs_mp8_ap2_FIXED_BASE,
            df_outputs_mp8_easiur_FIXED_BASE,
            df_outputs_mp8_inmap_FIXED_BASE
        ],
        subplot_positions=[(0, 0), (0, 1), (0, 2)],
        y_cols=[
            f'{scenario_prefix}{category}_health_npv_ap2_{cr_function}',
            f'{scenario_prefix}{category}_health_npv_easiur_{cr_function}',
            f'{scenario_prefix}{category}_health_npv_inmap_{cr_function}'
        ],
        hue_col=f'base_{category}_fuel',
        sharex=True,
        sharey=True,
        subplot_titles=[
            f'AP2 ({cr_function.upper()})', 
            f'EASIUR ({cr_function.upper()})', 
            f'InMAP ({cr_function.upper()})'
        ],
        x_labels=['', '', ''],
        y_labels=['Health NPV [2023 $USD]', '', ''],
        lower_percentile=lower_percentile,
        upper_percentile=upper_percentile,
        figure_size=(16, 6),
        show_outliers=False,
        show_xtick_labels=False
    )

    # Print summary statistics tables
    # ===== AP2 =====
    print_summary_stats(dataframes=[df_outputs_mp8_ap2_FIXED_BASE],
                        column_names=[f'{scenario_prefix}{category}_health_npv_ap2_{cr_function}'],
                        subplot_titles=[f'AP2 with {cr_function.upper()} CR-Function'])
    # ===== EASIUR =====
    print_summary_stats(dataframes=[df_outputs_mp8_easiur_FIXED_BASE],
                        column_names=[f'{scenario_prefix}{category}_health_npv_easiur_{cr_function}'],
                        subplot_titles=[f'EASIUR with {cr_function.upper()} CR-Function'])
    # ===== InMAP =====
    print_summary_stats(dataframes=[df_outputs_mp8_inmap_FIXED_BASE],
                        column_names=[f'{scenario_prefix}{category}_health_npv_inmap_{cr_function}'],
                        subplot_titles=[f'InMAP with {cr_function.upper()} CR-Function'])

    # Print positive percentage statistics
    print_positive_percentages_complete(
        dataframes=[
            df_outputs_mp8_ap2_FIXED_BASE,
            df_outputs_mp8_easiur_FIXED_BASE,
            df_outputs_mp8_inmap_FIXED_BASE
        ],
        column_names=[
            f'{scenario_prefix}{category}_health_npv_ap2_{cr_function}',
            f'{scenario_prefix}{category}_health_npv_easiur_{cr_function}',
            f'{scenario_prefix}{category}_health_npv_inmap_{cr_function}'
        ],
        subplot_titles=[
            f'AP2 ({cr_function.upper()})',
            f'EASIUR ({cr_function.upper()})',
            f'InMAP ({cr_function.upper()})'
        ],
        fuel_column=f'base_{category}_fuel'
    )

    # Store figure
    health_npv_figures[cr_function] = fig
    
    # Display
    display(fig)

## Climate Change Impact (SCC) and Tier 3 Adopters

### Space Heating - Progressive Impact of Climate Benefit Valuation

In [ ]:
scenario_prefix = 'iraRef_mp8_'
category = 'heating'
scc = 'central'
discount_rate = 'fixed_base'
lower_percentile = 0.5
upper_percentile = 99.5

print(f"""
===== FIGURE 7: CLIMATE BENEFIT IMPACT ON RETROFIT ADOPTION POTENTIAL (TIER 3) =====
- Retrofit Scenarios: {scenario_prefix} 
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate}
- Categories: {category}

Valid Range: {lower_percentile}th to {upper_percentile}th Percentile
""")

fig_heating_climate_scc_FIXED_BASE = create_subplot_grid_histogram(
    dataframes=[
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_BASE
        ],
    subplot_positions=[(0, 0), (0, 1), (0, 2), (0, 3)],  # 1x4 grid
    x_cols=[
        f'{scenario_prefix}{category}_private_npv_moreWTP_{discount_rate}',
        f'{scenario_prefix}{category}_total_npv_climateOnly_lower_{discount_rate}',
        f'{scenario_prefix}{category}_total_npv_climateOnly_central_{discount_rate}',
        f'{scenario_prefix}{category}_total_npv_climateOnly_upper_{discount_rate}'
    ],
    x_labels=['Private NPV [2023 $USD]'] + ['Total NPV [2023 $USD]'] * 3,
    y_labels=['Dwelling Units', '', '', ''],
    bin_number=40,  # Optional: number of bins for histogram
    lower_percentile=lower_percentile,    # Show nearly full range
    upper_percentile=upper_percentile,   # Show nearly full range
    subplot_titles=[
        'Private NPV Only\n37% Positive NPV',
        'SCC Lower Bound\n56% Positive NPV',
        'SCC Central Estimate\n78% Positive NPV',
        'SCC Upper Bound\n83% Positive NPV' 
    ],
    # suptitle=f'{category.title()}: Progressive Impact of Climate Benefit Valuation',
    figure_size=(20, 10),  # Wide format for 4 panels
    sharex=False,  # Keep different scales to show full distributions
    sharey=True,   # Same y-scale for comparison
    color_code=f'base_{category}_fuel'
)

print_positive_percentages_complete(
    df=df_outputs_mp8_inmap_FIXED_BASE,
    column_names=[
        f'{scenario_prefix}{category}_private_npv_moreWTP_{discount_rate}',
        f'{scenario_prefix}{category}_total_npv_climateOnly_lower_{discount_rate}',
        f'{scenario_prefix}{category}_total_npv_climateOnly_central_{discount_rate}',
        f'{scenario_prefix}{category}_total_npv_climateOnly_upper_{discount_rate}'
    ],
    subplot_titles=[
        f'Private NPV Only (Baseline), Discount Rate: {discount_rate}', 
        f'Lower Bound SCC (+ Climate), Discount Rate: {discount_rate}', 
        f'Central Estimate SCC (+ Climate), Discount Rate: {discount_rate}', 
        f'Upper Bound SCC (+ Climate), Discount Rate: {discount_rate}'
    ],
    fuel_column=f'base_{category}_fuel'
)

fig_heating_climate_scc_FIXED_BASE

# Adoption Rate Scenario Comparison

In [ ]:
from cmu_tare_model.adoption_potential.data_processing.visuals_adoption_potential import (
    create_multiIndex_adoption_df,
    print_adoption_decision_percentages,
    subplot_grid_adoption_vBar
)

if VERBOSE:

    print(f"""  
    ====================================================================================================================================================================
    ADOPTION POTENTIAL VISUALIZATION
    ====================================================================================================================================================================

    --------------------------------------------------------
    CREATE MULTI-INDEX DF FOR ADOPTION POTENTIAL
    --------------------------------------------------------
    visuals_adoption_potential.py file contains the documentation for the create_multiIndex_adoption_df function.

    --------------------------------------------------------
    VISUALIZE ADOPTION POTENTIAL SUBPLOT GRID
    --------------------------------------------------------
    visuals_adoption_potential.py file contains the documentation for the subplot_grid_adoption_vBar function.
        
    --------------------------------------------------------------------------------------------------------------------------------------------------------------------

    """)

## Space Heating - Basic (MP8), Moderate (MP9), Advanced (MP10) Retrofit


In [ ]:
# =============================================================================
# CREATE ADOPTION POTENTIAL DATAFRAMES (ALL COMBINATIONS)
# =============================================================================
# Creates DataFrames for all sensitivity combinations upfront.
# Structure: HEATING_ADOPTION_MI[discount_rate][mp][rcm][crf] = DataFrame
# Maybe reduce print output later if too verbose.

scc = 'central'
HEATING_MEASURE_PACKAGES = [8, 9, 10]

# Master dictionary to store all results
# Structure: [mp][discount_rate][rcm][crf] for consistency
ALL_HEATING_ADOPTION_MI = {
    mp: {
        discount_rate: {
            rcm: {crf: None for crf in CR_FUNCTIONS} 
            for rcm in RCM_MODELS
        }
        for discount_rate in PRIVATE_DISCOUNT_RATE_SHORT_KEYS
    }
    for mp in HEATING_MEASURE_PACKAGES
}

print("Creating adoption potential DataFrames...")

for menu_mp in HEATING_MEASURE_PACKAGES:
    print(f"\n{'='*80}")
    print(f"MEASURE PACKAGE {menu_mp}")
    print(f"{'='*80}")

    for discount_rate in PRIVATE_DISCOUNT_RATE_SHORT_KEYS:
        print(f"\nDiscount Rate: {discount_rate}")

        for rcm_model in RCM_MODELS:
            print(f"  RCM Model: {rcm_model.upper()}")
            for cr_function in CR_FUNCTIONS:
                # Direct dictionary access with short keys
                source_df = DATAFRAMES_BY_MP[menu_mp][discount_rate][rcm_model]
                
                df_mi = create_multiIndex_adoption_df(
                    df=source_df,
                    menu_mp=menu_mp,
                    category='heating',
                    scc=scc,
                    rcm_model=rcm_model,
                    cr_function=cr_function,
                    discount_rate=discount_rate
                )
                
                ALL_HEATING_ADOPTION_MI[menu_mp][discount_rate][rcm_model][cr_function] = df_mi
                print(f"    ✓ Created dataframes for {discount_rate} sensitivity: MP{menu_mp} | {rcm_model.upper()} | {cr_function.upper()} | Shape: {df_mi.shape}")

print(f"✓ Created {len(PRIVATE_DISCOUNT_RATE_SHORT_KEYS) * len(HEATING_MEASURE_PACKAGES) * len(RCM_MODELS) * len(CR_FUNCTIONS)} DataFrames")

In [ ]:
# =============================================================================
# VISUALIZATION CONFIGURATION
# =============================================================================
# Edit these values, then run the next cell to create the visualization.

# Discount rate: 'fixed_low', 'fixed_base', 'fixed_high', 'variable'
discount_rate = 'fixed_base'

# Health model parameters (typically keep these fixed)
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'

# =============================================================================
# ADOPTION POTENTIAL VISUALIZATION
# =============================================================================
# Subplot titles and labels for each measure package
MP_SUBTITLES = {
    8: "ASHP Only:\nNo IRA vs. IRA-Reference",
    9: "ASHP + Basic Enclosure:\nNo IRA vs. IRA-Reference",
    10: "ASHP + Enhanced Enclosure:\nNo IRA vs. IRA-Reference"
}

print(f"""
================================================================================
ADOPTION POTENTIAL VISUALIZATION
================================================================================
Discount Rate: {discount_rate}
SCC: {scc} | RCM: {rcm_model} | CRF: {cr_function}
""")

fig_adoption = subplot_grid_adoption_vBar(
    dataframes=[
        ALL_HEATING_ADOPTION_MI[8][discount_rate][rcm_model][cr_function],
        ALL_HEATING_ADOPTION_MI[9][discount_rate][rcm_model][cr_function], 
        ALL_HEATING_ADOPTION_MI[10][discount_rate][rcm_model][cr_function]
    ],
    scenarios_list=[
        [f'preIRA_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp8_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp9_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}'],
        [f'preIRA_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
         f'iraRef_mp10_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}']
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    filter_fuel=['Electricity', 'Natural Gas', 'Fuel Oil', 'Propane'],
    x_labels=["", "Fuel Type and Income Group (LMI: Low-to-Moderate-Income, MUI: Middle-to-Upper-Income)", ""],
    plot_titles=[MP_SUBTITLES[mp] for mp in HEATING_MEASURE_PACKAGES],
    y_labels=["Retrofit Adoption Potential (%)", "", ""],
    # suptitle=f"Space Heating Air-Source Heat Pump (ASHP) Retrofit Scenario Comparison\nClimate Sensitivity: SCC-{scc.upper()} | Health Sensitivity: {rcm_model.upper()}-{cr_function.upper()}",
    figure_size=(18, 12),
    sharey=True,
    x_tick_format="all"  # Use LMI/MUI classification for x-ticks
)

# =======================================================================================================
# PRINT ADOPTION DECISION PERCENTAGES FOR INMAP-ACS, FIXED-BASE DISCOUNT RATE
# =======================================================================================================
for i, menu_mp in enumerate(HEATING_MEASURE_PACKAGES):
    print_adoption_decision_percentages(
            dataframes=[
                ALL_HEATING_ADOPTION_MI[menu_mp][discount_rate][rcm_model][cr_function],
                ALL_HEATING_ADOPTION_MI[menu_mp][discount_rate][rcm_model][cr_function],
                ],
            scenario_names=[
                f'preIRA_mp{menu_mp}_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
                f'iraRef_mp{menu_mp}_heating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate}',
                ],
            source_dataframes=[
                DATAFRAMES_BY_MP[menu_mp][discount_rate][rcm_model],
                DATAFRAMES_BY_MP[menu_mp][discount_rate][rcm_model],
            ],
            category='heating',
            title=f"SPACE HEATING ADOPTION POTENTIAL: {discount_rate.upper()}", 
            subtitle=MP_SUBTITLES[menu_mp],
            print_header_key=True,
        )

fig_adoption

## Water Heating, Clothes Drying, and Cooking - Basic Retrofit (MP8)

In [ ]:
# =============================================================================
# CREATE ADOPTION POTENTIAL DFs FOR NON-HVAC CATEGORIES (ALL DISCOUNT RATES)
# =============================================================================
menu_mp = 8
scc = 'central'
CATEGORIES = ['waterHeating', 'clothesDrying', 'cooking']

print(f"""
=======================================================================================================
BASIC RETROFIT: MEASURE PACKAGE {menu_mp} (MP{menu_mp}) - NON-HVAC CATEGORIES
=======================================================================================================

Creating Multi-Index DataFrames for:
- Categories: {CATEGORIES}
- Discount Rates: {PRIVATE_DISCOUNT_RATE_SHORT_KEYS}
- RCM Models: {RCM_MODELS}
- CR Functions: {CR_FUNCTIONS}

Total combinations: {len(CATEGORIES)} categories × {len(PRIVATE_DISCOUNT_RATE_SHORT_KEYS)} rates × {len(RCM_MODELS)} RCMs × {len(CR_FUNCTIONS)} CRFs

""")

# Initialize nested dictionary to store results
# Structure: [category][discount_rate][rcm][crf]
MP8_NONHVAC_ADOPTION_MI = {
    category: {
        discount_rate: {
            rcm: {crf: None for crf in CR_FUNCTIONS}
            for rcm in RCM_MODELS
        }
        for discount_rate in PRIVATE_DISCOUNT_RATE_SHORT_KEYS
    }
    for category in CATEGORIES
}

# Category display names for pretty printing
CATEGORY_NAMES = {
    'waterHeating': 'Water Heating',
    'clothesDrying': 'Clothes Drying',
    'cooking': 'Cooking'
}

# Create all combinations using nested loops
for category in CATEGORIES:
    print(f"\n{'='*80}")
    print(f"CATEGORY: {CATEGORY_NAMES[category].upper()}")
    print(f"{'='*80}")
    
    for discount_rate_short in PRIVATE_DISCOUNT_RATE_SHORT_KEYS:
        print(f"\n  Discount Rate: {discount_rate_short}")
        
        for rcm_model in RCM_MODELS:
            print(f"    RCM Model: {rcm_model.upper()}")
            
            for cr_function in CR_FUNCTIONS:
                # Get the source DataFrame from LOADED DATA (not from adoption MI!)
                source_df = DATAFRAMES_MP8[discount_rate_short][rcm_model]
                
                # Create the multi-index adoption DataFrame
                df_mi = create_multiIndex_adoption_df(
                    df=source_df,
                    menu_mp=menu_mp,
                    category=category,
                    scc=scc,
                    rcm_model=rcm_model,
                    cr_function=cr_function,
                    discount_rate=discount_rate_short
                )
                
                # Store in nested dictionary
                MP8_NONHVAC_ADOPTION_MI[category][discount_rate_short][rcm_model][cr_function] = df_mi
                
                print(f"      ✓ {cr_function.upper()}: Shape {df_mi.shape}")

print(f"\n{'='*80}")
print(f"COMPLETE: Created {len(CATEGORIES) * len(PRIVATE_DISCOUNT_RATE_SHORT_KEYS) * len(RCM_MODELS) * len(CR_FUNCTIONS)} adoption DataFrames")
print(f"{'='*80}\n")

In [ ]:
# ====================================================================
# VISUALIZATION: Water Heating, Clothes Drying, Cooking - MP8
# ====================================================================
scc = 'central'
rcm_model = 'inmap'
cr_function = 'acs'
discount_rate_short = 'fixed_base'  # Or 'fixed_low', 'fixed_high', 'variable'

print(f"""
SENSITIVITY:
- SCC Climate Sensitivity: {scc}
- Discount Rate: {discount_rate_short}
- Health RCM Model: {rcm_model}
- Health CR Function: {cr_function}
""")

# Access using consistent structure: [category][discount_rate][rcm][crf]
fig_mp8_nonHVAC = subplot_grid_adoption_vBar(
    dataframes=[
        MP8_NONHVAC_ADOPTION_MI['waterHeating'][discount_rate_short][rcm_model][cr_function],
        MP8_NONHVAC_ADOPTION_MI['clothesDrying'][discount_rate_short][rcm_model][cr_function], 
        MP8_NONHVAC_ADOPTION_MI['cooking'][discount_rate_short][rcm_model][cr_function]
    ],
    scenarios_list=[
        [f'preIRA_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate_short}',
         f'iraRef_mp8_waterHeating_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate_short}'],
        [f'preIRA_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate_short}',
         f'iraRef_mp8_clothesDrying_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate_short}'],
        [f'preIRA_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate_short}',
         f'iraRef_mp8_cooking_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate_short}']
    ],
    subplot_positions=[(0, 0), (0, 1), (0, 2)],
    filter_fuel=['Electricity', 'Natural Gas', 'Fuel Oil', 'Propane'],
    x_labels=["", "Fuel Type and Income Group (LMI: Low-to-Moderate-Income, MUI: Middle-to-Upper-Income)", ""],
    plot_titles=[
        "Heat Pump Water Heater:\nNo IRA vs. IRA-Reference",
        "Heat Pump Clothes Dryer:\nNo IRA vs. IRA-Reference",
        "Electric Resistance Range:\nNo IRA vs. IRA-Reference"
    ],
    y_labels=["Retrofit Adoption Potential (%)", "", ""],
    figure_size=(18, 12),
    sharey=True,
    x_tick_format="all"
)

# Print statistics
for category in CATEGORIES:
    print_adoption_decision_percentages(
        dataframes=[
            MP8_NONHVAC_ADOPTION_MI[category][discount_rate_short][rcm_model][cr_function],
            MP8_NONHVAC_ADOPTION_MI[category][discount_rate_short][rcm_model][cr_function]
        ],
        scenario_names=[
            f'preIRA_mp8_{category}_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate_short}',
            f'iraRef_mp8_{category}_adoption_{scc}_{rcm_model}_{cr_function}_{discount_rate_short}'
        ],
        source_dataframes=[
            DATAFRAMES_MP8[discount_rate_short][rcm_model],
            DATAFRAMES_MP8[discount_rate_short][rcm_model]
        ],
        category=category,
        title=f"NON-HVAC ADOPTION: {discount_rate_short.upper()}",
        subtitle=CATEGORY_NAMES[category],
        print_header_key=True
    )

fig_mp8_nonHVAC

# SENSITIVITY ANALYSIS: Private Discount Rate and Adoption Feasibility (Retrofit Lifecycle Cost)

In [ ]:
# Discount Rate Sensitivity Analysis
category = 'heating'

fig_HEATING_preIRA_private_more_WTP_discount = create_subplot_grid_histogram(
    dataframes=[
        df_outputs_mp8_inmap_FIXED_LOW, 
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_HIGH,
        df_outputs_mp8_inmap_VARIABLE
    ],
    # dataframe_indices=[0, 0],
    subplot_positions=[(0, 0), (0, 1), (0, 2), (0, 3)],
    x_cols=[
        f'preIRA_mp8_{category}_private_npv_moreWTP_fixed_low',
        f'preIRA_mp8_{category}_private_npv_moreWTP_fixed_base',
        f'preIRA_mp8_{category}_private_npv_moreWTP_fixed_high',
        f'preIRA_mp8_{category}_private_npv_moreWTP_variable'
    ],
    x_labels=['Private NPV [$USD2023]',
              'Private NPV [$USD2023]',
              'Private NPV [$USD2023]',
              'Private NPV [$USD2023]'
    ],
    y_labels=['Dwelling units in Pre-IRA Scenario', '', '', ''],
    bin_number='auto',
    lower_percentile=lower_percentile,
    upper_percentile=upper_percentile,
    subplot_titles=['Fixed Discount Rate\n Low (2%)',
                    'Fixed Discount Rate\n Base (7%)',
                    'Fixed Discount Rate\n High (12%)',
                    'Variable Discount Rate\n Inverse to % AMI (7% to 45%)'],
    figure_size=(20, 10),  # Wide format for 4 panels
    sharex=False,  # Keep different scales to show full distributions
    sharey=True,   # Same y-scale for comparison
    color_code=f'base_{category}_fuel',
    show_legend=True
)

# Print comparison statistics
print("="*60)
print("Pre-IRA Scenario\nAdoption Feasibility under Different Discount Rate Assumptions")
print("="*60)

print_positive_percentages_complete(
    # df=df_outputs_mp8_ap2_FIXED_BASE,
    dataframes=[
        df_outputs_mp8_inmap_FIXED_LOW, 
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_HIGH,
        df_outputs_mp8_inmap_VARIABLE
    ],
    column_names=[
        f'preIRA_mp8_{category}_private_npv_moreWTP_fixed_low',
        f'preIRA_mp8_{category}_private_npv_moreWTP_fixed_base',
        f'preIRA_mp8_{category}_private_npv_moreWTP_fixed_high',
        f'preIRA_mp8_{category}_private_npv_moreWTP_variable'
    ],
    subplot_titles=['Fixed Discount Rate Low (2%)',
                    'Fixed Discount Rate Base (7%)',
                    'Fixed Discount Rate High (12%)',
                    'Variable Discount Rate Inverse to % AMI (7% to 45%)'],
    fuel_column=f'base_{category}_fuel'
)

fig_HEATING_preIRA_private_more_WTP_discount

In [ ]:
# Discount Rate Sensitivity Analysis
category = 'heating'

fig_HEATING_iraRef_private_more_WTP_discount = create_subplot_grid_histogram(
    dataframes=[
        df_outputs_mp8_inmap_FIXED_LOW, 
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_HIGH,
        df_outputs_mp8_inmap_VARIABLE
    ],
    # dataframe_indices=[0, 0],
    subplot_positions=[(0, 0), (0, 1), (0, 2), (0, 3)],
    x_cols=[
        f'iraRef_mp8_{category}_private_npv_moreWTP_fixed_low',
        f'iraRef_mp8_{category}_private_npv_moreWTP_fixed_base',
        f'iraRef_mp8_{category}_private_npv_moreWTP_fixed_high',
        f'iraRef_mp8_{category}_private_npv_moreWTP_variable'
    ],
    x_labels=['Private NPV [$USD2023]',
              'Private NPV [$USD2023]',
              'Private NPV [$USD2023]',
              'Private NPV [$USD2023]'
    ],
    y_labels=['Dwelling units in IRA-Reference Scenario', '', '', ''],
    bin_number='auto',
    lower_percentile=lower_percentile,
    upper_percentile=upper_percentile,
    subplot_titles=['Fixed Discount Rate\n Low (2%)',
                    'Fixed Discount Rate\n Base (7%)',
                    'Fixed Discount Rate\n High (12%)',
                    'Variable Discount Rate\n Inverse to % AMI (7% to 45%)'],
    figure_size=(20, 10),  # Wide format for 4 panels
    sharex=False,  # Keep different scales to show full distributions
    sharey=True,   # Same y-scale for comparison
    color_code=f'base_{category}_fuel',
    show_legend=True
)

# Print comparison statistics
print("="*60)
print("IRA-Reference Scenario\nAdoption Feasibility under Different Discount Rate Assumptions")
print("="*60)

print_positive_percentages_complete(
    # df=df_outputs_mp8_ap2_FIXED_BASE,
    dataframes=[
        df_outputs_mp8_inmap_FIXED_LOW, 
        df_outputs_mp8_inmap_FIXED_BASE,
        df_outputs_mp8_inmap_FIXED_HIGH,
        df_outputs_mp8_inmap_VARIABLE
    ],
    column_names=[
        f'iraRef_mp8_{category}_private_npv_moreWTP_fixed_low',
        f'iraRef_mp8_{category}_private_npv_moreWTP_fixed_base',
        f'iraRef_mp8_{category}_private_npv_moreWTP_fixed_high',
        f'iraRef_mp8_{category}_private_npv_moreWTP_variable'
    ],
    subplot_titles=['Fixed Discount Rate Low (2%)',
                    'Fixed Discount Rate Base (7%)',
                    'Fixed Discount Rate High (12%)',
                    'Variable Discount Rate Inverse to % AMI (7% to 45%)'],
    fuel_column=f'base_{category}_fuel'
)

fig_HEATING_iraRef_private_more_WTP_discount

# Model Runtime

In [ ]:
# Get the current datetime again
end_time = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# Calculate the elapsed time
elapsed_time = datetime.strptime(end_time, "%Y-%m-%d_%H-%M-%S") - datetime.strptime(start_time, "%Y-%m-%d_%H-%M-%S")

# Format the elapsed time
elapsed_seconds = elapsed_time.total_seconds()
elapsed_minutes = int(elapsed_seconds // 60)
elapsed_seconds = int(elapsed_seconds % 60)

# Print the elapsed time
print(f"The code took {elapsed_minutes} minutes and {elapsed_seconds} seconds to execute.")